# 09 — Operational Decision-Support Action Plan (Phases 1–4)

This executed notebook verifies and synthesizes the four operational phases implemented for the **Road to Safety** portfolio:
1. **Phase 1 (`corridor_pilot`)**: Corridor-level spatial + AADT exposure linkage (City of Toronto KSI + Centreline + SVC AADT, with Quebec MTQ highway comparator), Poisson/NB2 Safety Performance Functions, and Empirical Bayes (EB) hotspot screening.
2. **Phase 2 (`typology_severity`)**: Reintroducing person-level road-user (`P_USER`) and vehicle-level (`V_TYPE`) typologies across all 2,949,846 canonical NCDB collisions (1999–2021).
3. **Phase 3 (`operational_triage`)**: Cost-utility / Vickers-Elkin Net Benefit curves, 3-Tier operational triage architecture, and context-specific operating curves (rural high-speed dispatch vs. urban VRU traffic calming).
4. **Phase 4 (`monitor_prospective`)**: Open Canada CKAN release monitor, schema/sentinel validator, and covariate (PSI/JSD) + calibration (O/E, ECE) drift dashboard.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("..").resolve() if Path("../reports/tables").exists() else Path("projects/road_to_safety").resolve()
p1_summary = pd.read_csv(ROOT / "reports/tables/corridor_pilot_summary.csv")
p1_classes = pd.read_csv(ROOT / "reports/tables/corridor_pilot_by_road_class.csv")
p1_top = pd.read_csv(ROOT / "reports/tables/corridor_pilot_top_corridors.csv")
print("=== Phase 1: Corridor-Level Spatial + Exposure Summary ===")
print(p1_summary.T.to_string(header=False))
print("\n=== Exposure-Normalized Occurrence Rates by Functional Road Class ===")
print(p1_classes[["road_class", "monitored_segments", "monitored_length_km", "mean_aadt", "ksi_crashes", "fatal_crashes", "vru_ksi_crashes", "ksi_per_100m_vkt", "fatal_per_100m_vkt", "vru_ksi_per_100m_vkt"]].head(7).to_string(index=False))


=== Phase 1: Corridor-Level Spatial + Exposure Summary ===
jurisdiction                          City of Toronto (with Quebec MTQ Highway Comparator)
study_period                                                          2006-2023 (18 years)
total_ksi_collisions                                                                  7673
fatal_ksi_collisions                                                                  1073
valid_coord_collisions                                                                7672
snapped_within_50m_collisions                                                         7669
snapped_Within_50m_pct                                                               99.95
spatial_and_name_verified_collisions                                                  7559
median_snap_distance_m                                                                1.74
total_road_segments                                                                  50083
total_road_network_km          

In [2]:
print("=== Top 10 Empirical Bayes (EB) Priority Corridors (Toronto, >=2 km & >=0.5 100M VKT) ===")
cols = ["corridor_name", "monitored_length_km", "mean_aadt", "ksi_crashes", "fatal_crashes", "vru_ksi_crashes", "observed_ksi_per_100m_vkt", "eb_ksi_per_100m_vkt", "psi_excess_ksi"]
print(p1_top[cols].head(10).round(3).to_string(index=False))


=== Top 10 Empirical Bayes (EB) Priority Corridors (Toronto, >=2 km & >=0.5 100M VKT) ===
                  corridor_name  monitored_length_km  mean_aadt  ksi_crashes  fatal_crashes  vru_ksi_crashes  observed_ksi_per_100m_vkt  eb_ksi_per_100m_vkt  psi_excess_ksi
   Spadina Ave (Major Arterial)                2.092  15427.414           43              8               40                     21.515               13.982           8.520
St Clair Ave W (Major Arterial)                6.338  10401.326           64              8               45                     13.733               11.259           9.365
        Bay St (Major Arterial)                2.785  13876.401           37              4               29                     13.884               10.490           5.373
  Danforth Ave (Major Arterial)                5.059  16370.828           74              3               57                     13.169                9.941          11.297
   Danforth Rd (Major Arterial)              

In [3]:
p2_summary = pd.read_csv(ROOT / "reports/tables/typology_model_summary.csv")
p2_or = pd.read_csv(ROOT / "reports/tables/typology_odds_ratios.csv")
p2_sub = pd.read_csv(ROOT / "reports/tables/typology_subgroup_performance.csv")

print("=== Phase 2: Nested Severity Model Comparison (M0 vs M1 vs M2) ===")
print(p2_summary[["split", "model", "n", "fatal", "roc_auc", "pr_auc", "log_loss", "brier_score", "oe_ratio"]].round(5).to_string(index=False))
print("\n=== Adjusted Fatal Collision Odds Ratios (M2 Full Typology, Train 1999-2019) ===")
print(p2_or[["feature", "feature_group", "collisions_all_years", "fatal_collisions_all_years", "unadjusted_fatality_rate_pct", "odds_ratio"]].to_string(index=False))


=== Phase 2: Nested Severity Model Comparison (M0 vs M1 vs M2) ===
                    split              model       n  fatal  roc_auc  pr_auc  log_loss  brier_score  oe_ratio
          train_1999_2016 M0_conditions_only 2455890  39491  0.70465 0.03731   0.07806      0.01567   0.99695
     validation_2017_2019 M0_conditions_only  330376   5072  0.69455 0.03379   0.07565      0.01499   0.98064
     test_2020_2021_uncal M0_conditions_only  163580   3230  0.69584 0.04152   0.09276      0.01918   1.22745
test_2020_2021_recal_2019 M0_conditions_only  163580   3230  0.69584 0.04152   0.09277      0.01918   1.23252
          train_1999_2016     M1_plus_p_user 2455890  39491  0.72407 0.04350   0.07705      0.01562   0.99904
     validation_2017_2019     M1_plus_p_user  330376   5072  0.72318 0.04444   0.07418      0.01492   0.98329
     test_2020_2021_uncal     M1_plus_p_user  163580   3230  0.72151 0.05523   0.09093      0.01906   1.21870
test_2020_2021_recal_2019     M1_plus_p_user  163580 

In [4]:
print("=== Phase 2: Out-of-Time Test (2020-2021) Subgroup Calibration & Discrimination Recovery ===")
sub_test = p2_sub[p2_sub["split"] == "test_2020_2021"]
print(sub_test[["subgroup_label", "n", "fatal", "observed_fatality_rate_pct", "m0_mean_pred_pct", "m2_mean_pred_pct", "m0_roc_auc", "m2_roc_auc", "m0_pr_auc", "m2_pr_auc"]].to_string(index=False))


=== Phase 2: Out-of-Time Test (2020-2021) Subgroup Calibration & Discrimination Recovery ===
                               subgroup_label      n  fatal  observed_fatality_rate_pct  m0_mean_pred_pct  m2_mean_pred_pct  m0_roc_auc  m2_roc_auc  m0_pr_auc  m2_pr_auc
                               All Collisions 163580   3230                       1.975             1.609             1.632      0.6958      0.7628     0.0415     0.0747
                          Pedestrian Involved  17381    621                       3.573             1.270             2.738      0.6903      0.7276     0.0984     0.1347
                           Bicyclist Involved   8885     98                       1.103             1.257             0.831      0.6810      0.7576     0.0359     0.0475
                        Motorcyclist Involved  10144    467                       4.604             2.064             3.200      0.5878      0.6292     0.0701     0.1130
               Any Vulnerable Road User (VRU)  35977   11

In [5]:
p3_tiers = pd.read_csv(ROOT / "reports/tables/triage_tiered_operating_points.csv")
p3_bands = pd.read_csv(ROOT / "reports/tables/triage_mutually_exclusive_bands.csv")
p3_ctx = pd.read_csv(ROOT / "reports/tables/triage_context_specific_curves.csv")

print("=== Phase 3: 3-Tier Operational Triage vs Legacy Flat 80% Recall (Test 2020-2021) ===")
t_test = p3_tiers[p3_tiers["split"] == "test_2020_2021"]
print(t_test[["tier_id", "threshold", "alert_rate_pct", "precision_pct", "recall_pct", "fp_per_tp", "net_benefit_per_10k"]].to_string(index=False))
print("\n=== Mutually Exclusive Risk Bands (Test 2020-2021, M2 Full Typology) ===")
print(p3_bands.to_string(index=False))


=== Phase 3: 3-Tier Operational Triage vs Legacy Flat 80% Recall (Test 2020-2021) ===
                  tier_id  threshold  alert_rate_pct  precision_pct  recall_pct  fp_per_tp  net_benefit_per_10k
   Legacy_80pct_Recall_M0   0.011609          55.940          2.885      81.734      33.66               52.736
   Legacy_80pct_Recall_M2   0.011666          44.584          3.570      80.619      27.01               73.203
 Tier_1_Critical_Dispatch   0.088275           1.101         14.270       7.957       6.01                6.272
 Tier_2_Priority_Response   0.036237           9.376          7.257      34.458      12.78               33.258
Tier_3_Systemic_Watchlist   0.018215          28.061          4.468      63.498      21.38               71.767

=== Mutually Exclusive Risk Bands (Test 2020-2021, M2 Full Typology) ===
                     band  collisions  share_of_all_collisions_pct  fatal_collisions  share_of_all_fatals_pct  within_band_fatality_rate_pct  relative_risk_vs_routine
 

In [6]:
p4_drift = pd.read_csv(ROOT / "reports/tables/monitoring_annual_drift_summary.csv")
p4_cat = pd.read_csv(ROOT / "reports/tables/monitoring_catalog_status.csv")

print("=== Phase 4: Open Canada CKAN Catalog Status ===")
print(p4_cat[["ckan_package_id", "known_monitored_years", "verified_local_releases", "post_2021_microdata_releases_detected", "ingestion_status"]].to_string(index=False))
print("\n=== Phase 4: Historical Backtest (2017-2021) & Prospective 2022-2023 Drift Governance ===")
print(p4_drift[["release", "collisions", "observed_fatality_pct", "m2_mean_pred_pct", "m2_oe_ratio", "max_covariate_psi", "max_psi_feature", "alert_status"]].to_string(index=False))


=== Phase 4: Open Canada CKAN Catalog Status ===
                     ckan_package_id          known_monitored_years  verified_local_releases  post_2021_microdata_releases_detected                  ingestion_status
1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a 1999-2021 (23 annual releases)                       24                                      0 READY_WAITING_FOR_TC_2022_RELEASE

=== Phase 4: Historical Backtest (2017-2021) & Prospective 2022-2023 Drift Governance ===
                            release  collisions  observed_fatality_pct  m2_mean_pred_pct  m2_oe_ratio  max_covariate_psi max_psi_feature alert_status
                               2017      114412                 1.4841            1.5465       0.9596            0.00089          C_MNTH        GREEN
                               2018      111334                 1.5754            1.5533       1.0143            0.00093          C_MNTH        GREEN
                               2019      104630                 1.5483        